# Euclid DR1-Foundation compact peak pairs
Rerun the classical VIS/NISP J detector on public IRSA MER mosaics. The default is a two-tile shard. Select the entire pilot or held-out list in the parameters cell to replicate those counts. Files under `/content` are cleared when the Colab session ends. Download the result ZIP before closing the session.

The astronomical analysis uses deterministic image filtering, local peak statistics, WCS coordinate matching and an offset control. Coverage and byte counts refer to the selected image footprint and verified FITS row ranges.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys
from zipfile import ZipFile

workspace = Path("/content")
archive = workspace / "Euclid_DR1F_Reproducibility.zip"
study = workspace / "edf-north-compact-doubles"
if archive.exists():
    with ZipFile(archive) as bundle:
        bundle.extractall(workspace)
else:
    repository = workspace / "euclid-research"
    if not repository.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                        "--branch", "research/euclid-dr1-foundation-day01-20261114",
                        "https://github.com/Biswajit1999/daily-astro-notebooks.git", str(repository)],
                       check=True)
    subprocess.run(["git", "-C", str(repository), "sparse-checkout", "set", "euclid-dr1-foundation/edf-north-compact-doubles"], check=True)
    study = repository / "euclid-dr1-foundation/edf-north-compact-doubles"
assert (study / "search_doubles.py").is_file()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "astropy>=7.2,<9",
                "scipy>=1.13", "numpy>=2.0"], check=True)


In [ ]:
pilot = list(range(101829775, 101829781))
holdout = [101829781, 101829782, *range(101830399, 101830409)]
tiles = holdout[:2]
workers = 2
cache = workspace / "euclid-cache"
output = workspace / ("euclid-results-" + "-".join(map(str, tiles)))
environment = dict(os.environ, OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1")

subprocess.run([sys.executable, str(study / "search_doubles.py"),
                "--cache", str(cache), "--output", str(output), "--workers", str(workers),
                "--tiles", *map(str, tiles)], env=environment, check=True)


In [ ]:
import json
import shutil
from google.colab import files

coverage_file = output / "footprint_summary.json"
subprocess.run([sys.executable, str(study / "cohort_summary.py"),
                "--pilot", str(output), "--pilot-cache", str(cache),
                "--output", str(coverage_file)], env=environment, check=True)
summary = json.loads((output / "search_summary.json").read_text())
coverage = json.loads(coverage_file.read_text())["total"]
missing = [tile["tile"] for tile in summary["tiles"] if "vis" not in tile or "j" not in tile]
print("Missing or unusable tiles", missing)
print("Repeaters / testable", summary["j_repeat"], "/", summary["j_tested"])
print("North-shift passes / testable", summary["shift_repeat"], "/", summary["shift_tested"])
print("Common footprint and byte counts", coverage)
files.download(shutil.make_archive(str(output), "zip", output))


For a larger run, use a separate output directory for each disjoint tile shard, and preserve failures. Check the union of observed WCS footprints before comparing area to a release percentage. An all-product 170 TB job requires distributed processing near the archive. Colab is suitable for verifying small shards.
